# ElectronX market data: worked examples

A tour of `electron_x_data_collector`, from pulling data to first-pass signal research.

1. Setup and the dataset catalogue
2. Retrieving data: L1, L2, trades, quotes, filters, caching, streaming
3. Symbols and constants
4. Frame helpers: mid/spread, long-format book, validated row models
5. Research on L1: quote state, forward moves, information coefficient, trade markouts

**Before running:** put `ELECTRONX_USER` / `ELECTRONX_TOKEN` in `.env` at the repo root, and make sure
`src/electron_x_data_collector/_vendor/liberator.py` exists (see README). The cells below use one
ERCOT morning so they finish quickly; widen the dates once you know what you're looking for.

## 1. Setup and the dataset catalogue

In [ ]:
import pandas as pd

from electron_x_data_collector import (
    LOT_SIZE_MWH,
    MARKET_LAUNCH_DATES,
    TICK_SIZE,
    ElectronXClient,
    Symbol,
    add_forward_mids,
    add_mid_spread,
    add_quote_state,
    add_symbol_columns,
    book_to_long,
    ic_summary,
    information_coefficient,
    iter_models,
    summarize,
    trade_markouts,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Credentials come from .env. Finished days are cached as Parquet, so re-running is instant.
ex = ElectronXClient(cache_dir="../.cache/electronx")
ex

In [ ]:
ex.datasets()

In [ ]:
# Coverage and schema for each dataset (pydantic models)
for name, info in ex.dataset_info().items():
    s = info.stats
    print(f"{name}: {s.first_date} -> {s.last_date}, {s.row_count:,} rows, {s.symbol_count:,} symbols")

In [ ]:
pd.DataFrame([c.model_dump() for c in ex.schema("L1 TAQ")])[["name", "type", "description"]]

## 2. Retrieving data

Every query covers `[back_to, as_of)`. Naive times are New York time and a bare date means midnight.
The filters (`symbols`, `markets`, `products`, `event_type`, `fields`, `where`) run on the server, so
filter as much as you can: an unfiltered pull of every symbol is slow.

### L1: top of book plus trades

In [ ]:
START, END = "2026-10-15 09:00", "2026-10-15 13:00"

l1 = ex.l1(START, END, markets="ERCOT", parse_symbols=True)
print(f"{len(l1):,} rows, {l1.symbol.nunique()} symbols")
print(l1.event_type.value_counts().to_dict())
l1.head()

In [ ]:
# Columns come back typed: tz-aware times, float prices (USD/MWh), nullable Int64 sizes (lots), categoricals
l1.dtypes

In [ ]:
# parse_symbols=True added product / market / location / expiry columns
l1.groupby(["location", "expiry_hour"], observed=True).size().unstack(fill_value=0)

### Trades and quotes only

In [ ]:
trades = ex.trades(START, END, markets="ERCOT")   # event_type == 'trade', trimmed to trade columns
quotes = ex.quotes(START, END, markets="ERCOT")   # event_type == 'quote'
print(len(trades), "trades,", len(quotes), "quotes")
trades.head()

### Other filters

`products`, a list of `markets`, a column subset with `fields`, and a raw SQL `where` clause.
`where` is AND-ed onto the other filters.

In [ ]:
hub = ex.l1(
    START, END,
    markets=["ERCOT"],
    products="BFUT",
    where="contains(symbol, 'HB_NORTH')",
    fields=["timestamp", "muts", "symbol", "event_type", "best_bid", "best_ask"],
)
hub.head()

### L2: 10-level book

L2 is heavier, so pick one symbol: here, the busiest one in the L1 pull.

In [ ]:
busiest = l1.symbol.value_counts().index[0]
print(busiest)

l2 = ex.l2(START, END, symbols=busiest)
print(f"{len(l2):,} rows")
l2[["timestamp", "event_type", "bid_px_1", "bid_qty_1", "ask_px_1", "ask_qty_1", "bid_px_2", "ask_px_2"]].head()

### Streaming and validated events

`iter_batches` streams raw Arrow batches from one request, with no cache, for pulls too big to hold in memory.
`iter_events` yields each row as a validated `L1Event` / `L2Event`.

In [ ]:
n_rows = 0
for batch in ex.iter_batches("L1 TAQ", "2026-10-15 10:00", "2026-10-15 10:15", markets="ERCOT"):
    n_rows += batch.num_rows
print("streamed rows:", n_rows)

In [ ]:
events = ex.iter_events("L1 TAQ", "2026-10-15 10:00", "2026-10-15 10:05", symbols=busiest)
for ev in list(events)[:5]:
    print(ev.timestamp.time(), ev.event_type.value, ev.best_bid, ev.best_ask, ev.mid, ev.is_trade)

## 3. Symbols and constants

In [ ]:
s = Symbol.parse(busiest)
print(s.product_code, s.market, s.location, s.expiry)
print("tick size:", TICK_SIZE, "USD/MWh | lot:", LOT_SIZE_MWH, "MWh")
MARKET_LAUNCH_DATES

In [ ]:
# add_symbol_columns does the same parsing for a whole frame (when parse_symbols=False was used)
add_symbol_columns(trades)[["symbol", "product", "market", "location", "expiry"]].drop_duplicates().head()

## 4. Frame helpers

In [ ]:
# Mid and spread straight from each row's top of book (quote rows only carry a book)
add_mid_spread(quotes).dropna(subset=["mid"])[["timestamp", "symbol", "best_bid", "best_ask", "mid", "spread"]].head()

In [ ]:
# Wide L2 -> one row per (event, side, level): handy for depth profiles
book = book_to_long(l2)
book.groupby(["side", "level"], observed=True)["qty"].mean().unstack("side")

In [ ]:
# Validate rows from a frame you already have; L2Event exposes .bids / .asks as BookLevel lists
ev = next(e for e in iter_models(l2, "L2 TAQ") if e.bids and e.asks)
ev.bids[:3], ev.asks[:3]

## 5. Research on L1

All moves are in **ticks** (0.25 USD/MWh), not percent, since power prices can be near zero or negative.

The pull above mixes BFUT futures with BOPT options. Option mids don't belong in the same return statistics,
so the research below uses BFUT only.

### Prevailing quote state

`add_quote_state` gives every row the latest quote at or before it. A trade row therefore sees the book
just before the trade. A quote with an empty side, or a locked or crossed book, has `mid = NaN`.

In [ ]:
fut = l1[l1["product"] == "BFUT"]
q = add_quote_state(fut)
q.loc[q.symbol == busiest, ["timestamp", "event_type", "bid", "ask", "bid_qty", "ask_qty", "mid", "spread", "imbalance"]].head(10)

In [ ]:
q.groupby("location", observed=True)[["spread", "imbalance"]].describe().round(2)

### Forward mid moves

A horizon is either a time (`"10s"`, measured on matching-engine time `muts`) or an event count (`int`).
Forward values are NaN past a symbol's last event, so nothing is extrapolated.

In [ ]:
H = ["1s", "10s", "60s", "300s"]
q = add_forward_mids(q, H + [5])
summarize(q, [f"ret_{h}" for h in H] + ["ret_5ev"])

### Does the signal predict the move? Information coefficient

`information_coefficient` computes a Spearman IC per group. Treat each group (a day, or a symbol here,
since this is a single morning) as one observation. Overlapping horizons make per-event statistics look
much stronger than they are.

In [ ]:
ic = information_coefficient(q, "imbalance", H, by="symbol", min_obs=50)
ic_summary(ic)

### Trade markouts

`trade_markouts` returns one row per trade, from the aggressor's side, in ticks:

* `eff_spread`: what the aggressor paid to cross (a passive fill earns it)
* `markout_<h>`: the aggressor's PnL per lot at the horizon. The passive side earns the negative.

In [ ]:
mk = trade_markouts(fut, H)
mk.head()

In [ ]:
cols = ["eff_spread", *[f"markout_{h}" for h in H]]
summarize(mk, cols)

In [ ]:
# By aggressor side, and weighted by trade size
summarize(mk, cols, by="aggressor_side", weight="trade_qty")

### Next steps

* Widen to several weeks (`ex.l1("2026-09-01", "2026-10-16", markets="ERCOT")`) and use `by="trade_date"` for the IC.
* Try other signals: add a column to `q` and pass its name to `information_coefficient`.
* Compare passive edge (`eff_spread / 2 - markout`) against fees before building a full event-driven backtest.